## Creating a reasoning model using unsloth

This notebook makes use of unsloth to finetune a llama 3.1 model into a reasoning model. I would recommend using the unsloth library compared to just using the huggingface library as it requires less memory and is faster.

Adapted from unsloth notebooks, if something is broken check on:
https://unsloth.ai/

In [1]:
%%capture
import os
!pip install --no-deps unsloth vllm
!pip install --no-deps unsloth vllm
# [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
# Skip restarting message in Colab
import sys, re, requests; modules = list(sys.modules.keys())
for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

# vLLM requirements - vLLM breaks Colab due to reinstalling numpy
f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
with open("vllm_requirements.txt", "wb") as file:
    file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
!pip install -r vllm_requirements.txt

### Add lora to base model and patch with Unsloth

In [2]:
# from unsloth import FastLanguageModel
# import torch
# max_seq_length = 1024
# lora_rank = 32

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "Qwen/Qwen2.5-3B-Instruct",
#     max_seq_length = max_seq_length,
#     load_in_4bit = True,
#     fast_inference = True,
#     max_lora_rank = lora_rank,
#     gpu_memory_utilization = 0.6,
# )

# model = FastLanguageModel.get_peft_model(
#     model,
#     r = lora_rank,
#     target_modules = [
#         "q_proj", "k_proj", "v_proj", "o_proj",
#         "gate_proj", "up_proj", "down_proj",
#     ],
#     lora_alpha = lora_rank,
#     use_gradient_checkpointing = "unsloth",
#     random_state = 3407,
# )

# ---- hard-disable vLLM path + import order ----
# Disable vLLM to avoid cudagraph FULL issues on T4
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"   # or: os.environ["UNSLOTH_FORCE_NO_VLLM"] = "1"

import unsloth
from unsloth import FastLanguageModel
import torch

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

max_seq_length = 1024
lora_rank = 32
FORCE_GPU = {"": "cuda:0"}  # keep everything on GPU for 4-bit

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name              = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length          = max_seq_length,
    load_in_4bit            = True,
    fast_inference          = False,          # no vLLM
    max_lora_rank           = lora_rank,
    gpu_memory_utilization  = 0.6,
    device_map              = FORCE_GPU,      # avoid CPU/disk offload
    low_cpu_mem_usage       = True,
    torch_dtype             = torch.float16,  # explicit dtype
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = lora_rank,
    target_modules             = ["q_proj","k_proj","v_proj","o_proj",
                                  "gate_proj","up_proj","down_proj"],
    lora_alpha                 = lora_rank,
    use_gradient_checkpointing = "unsloth",
    random_state               = 3407,
)

print("Model ready ✅ on", next(model.parameters()).device)





🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 09-27 06:16:01 [__init__.py:216] Automatically detected platform cuda.
ERROR 09-27 06:16:02 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8


    PyTorch 2.6.0+cu124 with CUDA 1204 (you have 2.8.0+cu126)
    Python  3.12.9 (you have 3.12.11)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


Switching to PyTorch attention since your Xformers is broken.

Unsloth: Xformers was not installed correctly.
Please install xformers separately first.
Then confirm if it's correctly installed by running:
python -m xformers.info

Longer error message:
xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.6.0+cu124 with CUDA 1204 (you have 2.8.0+cu126)
    Python  3.12.9 (you have 3.12.11)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.9.9: Fast Qwen2 patching. Transformers: 4.56.1. vLLM: 0.10.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http:

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Unsloth 2025.9.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Model ready ✅ on cuda:0


### Data Prep
<a name="Data"></a>

Uses [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) reward functions.

In [3]:
import re
from datasets import load_dataset, Dataset

# Load and prep dataset
SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

# uncomment middle messages for 1-shot prompting
def get_gsm8k_questions(split = "train") -> Dataset:
    data = load_dataset('openai/gsm8k', 'main')[split] # type: ignore
    data = data.map(lambda x: { # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']}
        ],
        'answer': extract_hash_answer(x['answer'])
    }) # type: ignore
    return data # type: ignore

dataset = get_gsm8k_questions()

# Reward functions
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}")
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

def int_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [5]:
max_prompt_length = 256

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1,
    num_generations = 6,
    max_prompt_length = max_prompt_length,
    max_completion_length = max_seq_length - max_prompt_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 100,
    save_steps = 100,
    max_grad_norm = 0.1,
    report_to = "none",
    output_dir = "outputs",
)

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 6


In [6]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        xmlcount_reward_func,
        soft_format_reward_func,
        strict_format_reward_func,
        int_reward_func,
        correctness_reward_func,
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,473 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 6 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (6 x 1 x 1) = 6
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)


-------------------- Question:
A concert ticket costs $40. Mr. Benson bought 12 tickets and received a 5% discount for every ticket bought that exceeds 10. How much did Mr. Benson pay in all? 
Answer:
476 
Response:
<reasoning>
Mr. Benson bought 12 concert tickets, so 2 of the tickets were eligible for a 5% discount because he bought more than 10 tickets. The cost of a ticket is $40. The first 10 tickets cost $40 each, and the 12th ticket gets a 5% discount.

Calculate the cost for the first 10 tickets:
10 tickets * $40 per ticket = $400

Next, calculate the discounted price for the 12th ticket:
Original price = $40
Discount = 5% of $40 = 0.05 * 40 = $2
Discounted price = $40 - $2 = $38

Now add the cost of the first 10 tickets with the discounted price of the 12th ticket:
Total cost = $400 + $38 = $438

</reasoning>
<answer>
Mr. Benson paid a total of $438 for the 12 concert tickets.</answer>
 
Extracted:
Mr. Benson paid a total of $438 for the 12 concert tickets.


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / xmlcount_reward_func / mean,rewards / xmlcount_reward_func / std,rewards / soft_format_reward_func / mean,rewards / soft_format_reward_func / std,rewards / strict_format_reward_func / mean,rewards / strict_format_reward_func / std,rewards / int_reward_func / mean,rewards / int_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std
1,0.000000,-0.431167,0.109596,271.666687,228.000000,303.000000,0.000000,271.666687,228.000000,303.000000,0,0,0,0,0,0.000000,-0.431167,0.109596,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,-0.000000,-0.834333,1.002841,476.500000,342.000000,648.000000,0.000000,476.500000,342.000000,648.000000,No Log,No Log,No Log,No Log,No Log,0.000000,-0.917667,0.883453,0.000000,0.000000,0.000000,0.000000,0.083333,0.204124,0.000000,0.000000
3,-0.000000,-0.708667,0.216983,341.000000,272.000000,421.000000,0.000000,341.000000,272.000000,421.000000,No Log,No Log,No Log,No Log,No Log,0.000000,-0.708667,0.216983,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,-0.376333,0.234234,303.166687,194.000000,483.000000,0.000000,303.166687,194.000000,483.000000,No Log,No Log,No Log,No Log,No Log,0.000034,-0.376333,0.234234,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,-0.000000,-0.255500,0.147808,252.166672,203.000000,330.000000,0.000000,252.166672,203.000000,330.000000,No Log,No Log,No Log,No Log,No Log,0.000043,-0.255500,0.147808,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,0.000000,-0.176167,0.853143,364.833344,311.000000,452.000000,0.000000,364.833344,311.000000,452.000000,No Log,No Log,No Log,No Log,No Log,0.000034,-0.592833,0.217607,0.000000,0.000000,0.000000,0.000000,0.083333,0.204124,0.333333,0.816497
7,0.000000,-0.414500,0.192519,312.333344,256.000000,429.000000,0.000000,312.333344,256.000000,429.000000,No Log,No Log,No Log,No Log,No Log,0.000231,-0.414500,0.192519,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,-0.000000,-0.510667,1.195255,350.333344,275.000000,426.000000,0.000000,350.333344,275.000000,426.000000,No Log,No Log,No Log,No Log,No Log,0.000035,-0.927333,0.355857,0.000000,0.000000,0.000000,0.000000,0.083333,0.204124,0.333333,0.816497
9,-0.000000,-0.431333,0.247560,272.166687,174.000000,317.000000,0.000000,272.166687,174.000000,317.000000,No Log,No Log,No Log,No Log,No Log,0.000036,-0.431333,0.247560,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
10,-0.000000,-0.549667,0.104170,323.000000,277.000000,357.000000,0.000000,323.000000,277.000000,357.000000,No Log,No Log,No Log,No Log,No Log,0.000037,-0.549667,0.104170,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


Unsloth: Will smartly offload gradients to save VRAM!
-------------------- Question:
Jane is trying to decide whether to buy a house or a trailer. A house costs $480,000 and a trailer costs $120,000. Each loan will be paid in monthly installments over 20 years. How much more is the monthly payment on the house compared to the trailer? 
Answer:
1500 
Response:
<reasoning>
First, let determine the monthly payments for both loan scenarios (house and trailer) by using the formula for calculating monthly payments on a fixed-rate loan over a period of time. The formula is:

\[ \text{Monthly Payment} = \frac{P \times r \times (1 + r)^n}{(1 + r)^n - 1} \]

Where:
- \( P \) is the principal (loan amount).
- \( r \) is the monthly interest rate.
- \( n \) is the number of payments (loan term in months).

Since the interest rate is not given, we'll assume a standard situation where the monthly payments are made straight into a sinking fund for the house and a down payment for the trailer; simplif

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / xmlcount_reward_func / mean,rewards / xmlcount_reward_func / std,rewards / soft_format_reward_func / mean,rewards / soft_format_reward_func / std,rewards / strict_format_reward_func / mean,rewards / strict_format_reward_func / std,rewards / int_reward_func / mean,rewards / int_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std
1,0.000000,-0.431167,0.109596,271.666687,228.000000,303.000000,0.000000,271.666687,228.000000,303.000000,0,0,0,0,0,0.000000,-0.431167,0.109596,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,-0.000000,-0.834333,1.002841,476.500000,342.000000,648.000000,0.000000,476.500000,342.000000,648.000000,No Log,No Log,No Log,No Log,No Log,0.000000,-0.917667,0.883453,0.000000,0.000000,0.000000,0.000000,0.083333,0.204124,0.000000,0.000000
3,-0.000000,-0.708667,0.216983,341.000000,272.000000,421.000000,0.000000,341.000000,272.000000,421.000000,No Log,No Log,No Log,No Log,No Log,0.000000,-0.708667,0.216983,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,-0.376333,0.234234,303.166687,194.000000,483.000000,0.000000,303.166687,194.000000,483.000000,No Log,No Log,No Log,No Log,No Log,0.000034,-0.376333,0.234234,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,-0.000000,-0.255500,0.147808,252.166672,203.000000,330.000000,0.000000,252.166672,203.000000,330.000000,No Log,No Log,No Log,No Log,No Log,0.000043,-0.255500,0.147808,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,0.000000,-0.176167,0.853143,364.833344,311.000000,452.000000,0.000000,364.833344,311.000000,452.000000,No Log,No Log,No Log,No Log,No Log,0.000034,-0.592833,0.217607,0.000000,0.000000,0.000000,0.000000,0.083333,0.204124,0.333333,0.816497
7,0.000000,-0.414500,0.192519,312.333344,256.000000,429.000000,0.000000,312.333344,256.000000,429.000000,No Log,No Log,No Log,No Log,No Log,0.000231,-0.414500,0.192519,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,-0.000000,-0.510667,1.195255,350.333344,275.000000,426.000000,0.000000,350.333344,275.000000,426.000000,No Log,No Log,No Log,No Log,No Log,0.000035,-0.927333,0.355857,0.000000,0.000000,0.000000,0.000000,0.083333,0.204124,0.333333,0.816497
9,-0.000000,-0.431333,0.247560,272.166687,174.000000,317.000000,0.000000,272.166687,174.000000,317.000000,No Log,No Log,No Log,No Log,No Log,0.000036,-0.431333,0.247560,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
10,-0.000000,-0.549667,0.104170,323.000000,277.000000,357.000000,0.000000,323.000000,277.000000,357.000000,No Log,No Log,No Log,No Log,No Log,0.000037,-0.549667,0.104170,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


-------------------- Question:
Matthew, the baker, arrives at work early every morning to make his famous caramel-apple coffee cakes for the day.  It usually takes Matthew 1 hour to assemble ingredients, 1.5 hours to bake the cakes, and another 1 hour to decorate each cake with cinnamon sprinkles.  One day, Matthew's oven failed to reach the correct temperature and it took twice as long for the cakes to bake as they usually take.  On that day, how long did it take, in hours, for Matthew to make his famous caramel-apple coffee cakes? 
Answer:
5 
Response:
<reasoning>
Matthew's usual times for each process are:
- 1 hour to assemble ingredients,
- 1.5 hours to bake (twice as long so 3 hours),
- 1 hour to decorate.

If the oven failed and the baking took twice as long, that means he spent 3 * 2 = 6 hours baking the cakes. On this day, Matthew's total time to make the cake would be the sum of the time to assemble the ingredients, the actual time to bake the cakes, and the time to decorate. 

TrainOutput(global_step=100, training_loss=0.00013512650603869858, metrics={'train_runtime': 4997.8909, 'train_samples_per_second': 0.12, 'train_steps_per_second': 0.02, 'total_flos': 0.0, 'train_loss': 0.00013512650603869858})

<a name="Inference"></a>
### Inference
Try inference before adding lora

In [10]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "How many r's are in strawberry?"},
], tokenize = False, add_generation_prompt = True)

# from vllm import SamplingParams
# sampling_params = SamplingParams(
#     temperature = 0.8,
#     top_p = 0.95,
#     max_tokens = 1024,
# )
output = model.generate(
    tokenizer(text, return_tensors="pt").input_ids.to(model.device),
    temperature = 0.8,
    top_p = 0.95,
    max_new_tokens = 1024,
    # lora_request = None, # This is for vLLM, not needed for transformers generate
)[0]

output = tokenizer.decode(output, skip_special_tokens=True)

output

'system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.\nuser\nHow many r\'s are in strawberry?\nassistant\nThere are no letters \'r\' in the word "strawberry".'

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [14]:
model.save_pretrained("grpo_saved_lora")

Now we load the LoRA and test:

In [17]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "মারির কাছে ৩টা আপেল আছে। সে আরো ২টা কিনলো। মোট কত আপেল??"},
], tokenize = False, add_generation_prompt = True)

# from vllm import SamplingParams # No longer needed
# sampling_params = SamplingParams(
#     temperature = 0.8,
#     top_p = 0.95,
#     max_tokens = 1024,
# ) # No longer needed

output = model.generate(
    tokenizer(text, return_tensors="pt").input_ids.to(model.device),
    temperature = 0.8, # Pass parameters directly
    top_p = 0.95, # Pass parameters directly
    max_new_tokens = 1024, # Use max_new_tokens instead of max_tokens
)[0]

output = tokenizer.decode(output, skip_special_tokens=True)

output

'system\n\nRespond in the following format:\n<reasoning>\n...\n</reasoning>\n<answer>\n...\n</answer>\n\nuser\nমারির কাছে ৩টা আপেল আছে। সে আরো ২টা কিনলো। মোট কত আপেল??\nassistant\n<reasoning>\nমারি মুখে থাকা আপেলগুলি ওকে বর্ণনা দিয乱了, তার নতুন কিনা আপেলগুলিও মুখে থাকা আপেলগুলিকে মিশিয়ে গিয়ে হিসাব রাখতে হবে। মারির মুখে থাকা আপেলগুলিতে ৩টা আছে এবং তার নতুন কিনা ২টা আছে, যা মারির কাছে পৌঝা হয়েছে। যদিও মারির কাছে আপেল আছে, কিনা বা আছে না চিহ্নিত করা নেই অন্যথায় তার মোট আপেল জীবিকা হিসাব রাখতে লজিকালভাবে এই টেক্স্ট ব্যাখ্যা করা যায় না। তাই মোট আপেল সংখ্যাটি প্রস্তাব রাখতে পারেন না।\n</reasoning>\n<answer>\nএই টেক্স্ট ব্যাখ্যায় মারির মোট আপেল সংখ্যা প্রকাশ করা যায় নেই। মারি মুখে থাকা আপেলগুলিকে পরে আরও নতুন আপেলগুলি পরিষ্কার করে ধরা হতে হবে। বর্ণনা নেই, নতুন আপেলগুলি পুনরাবৃত্তির জন্য আমার জন্য কোনও সূত্রায়মান উত্তর পাওয়া যায় নেই।\n</answer>'

In [18]:
print(output)

system

Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>

user
মারির কাছে ৩টা আপেল আছে। সে আরো ২টা কিনলো। মোট কত আপেল??
assistant
<reasoning>
মারি মুখে থাকা আপেলগুলি ওকে বর্ণনা দিয乱了, তার নতুন কিনা আপেলগুলিও মুখে থাকা আপেলগুলিকে মিশিয়ে গিয়ে হিসাব রাখতে হবে। মারির মুখে থাকা আপেলগুলিতে ৩টা আছে এবং তার নতুন কিনা ২টা আছে, যা মারির কাছে পৌঝা হয়েছে। যদিও মারির কাছে আপেল আছে, কিনা বা আছে না চিহ্নিত করা নেই অন্যথায় তার মোট আপেল জীবিকা হিসাব রাখতে লজিকালভাবে এই টেক্স্ট ব্যাখ্যা করা যায় না। তাই মোট আপেল সংখ্যাটি প্রস্তাব রাখতে পারেন না।
</reasoning>
<answer>
এই টেক্স্ট ব্যাখ্যায় মারির মোট আপেল সংখ্যা প্রকাশ করা যায় নেই। মারি মুখে থাকা আপেলগুলিকে পরে আরও নতুন আপেলগুলি পরিষ্কার করে ধরা হতে হবে। বর্ণনা নেই, নতুন আপেলগুলি পুনরাবৃত্তির জন্য আমার জন্য কোনও সূত্রায়মান উত্তর পাওয়া যায় নেই।
</answer>


## Saving

### Save lora adapter

This is both useful for inference and if you want to load the model again

In [22]:
model.push_to_hub(
    "shalhamucha/Qwen2.5-3B-Instruct-Reasoning-for-math",
    tokenizer,
    token = userdata.get('HF_ACCESS_TOKEN')
)

README.md:   0%|          | 0.00/612 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  546kB /  240MB            

Saved model to https://huggingface.co/shalhamucha/Qwen2.5-3B-Instruct-Reasoning-for-math


### Merge model with lora weights and save to gguf

You can then do inference locally with Ollama or llama.cpp

##### Popular quantization methods

- **q4_k_m**  
  4bit quantization. Low memory. All models you pull with ollama uses this quantization.
- **q8_0**  
  8bit quantization. Medium memory.
- **f16**  
  16 bit quantization. A lot of models are already in 16 bit so then no quantization happens
- **not_quantized**  
  Often same as f16.

In [2]:
from google.colab import userdata

# model.merge_and_unload_lora() # Remove this line
model.push_to_hub_gguf(
    "shalhamucha/Qwen2.5-3B-Reasoning-math-GGUF",
    tokenizer,
    token = userdata.get('HF_ACCESS_TOKEN')
)

NameError: name 'model' is not defined